# 00 – One-time setup: the `oldman_isce2` environment

Run this notebook **once**, with the normal OpenSARlab kernel (the one called **Python 3**). It takes about 20–40 minutes.

It installs, in `~/.local/envs/oldman_isce2`:
- **ISCE2 2.6.5** (topsStack), **MintPy 1.6.4**, **burst2safe 2.0.3**, **s1_orbits 0.2.0**, **sardem 0.14.0**, **asf_search 14.1.0**
- **MiaplPy** (pinned to one tested version). It makes the `slcStack.h5` and `geometryRadar.h5` files.

Every version is pinned in `oldman_isce2_env.yml`, so you get exactly the environment that was tested.

**Before you start:** put `oldman_isce2_env.yml` in the **same folder** as this notebook.

Run the cells one by one from top to bottom (Shift + Enter).

In [1]:
import os, shutil, subprocess, sys
from pathlib import Path

HERE = Path.cwd()
YML = HERE / 'oldman_isce2_env.yml'
PREFIX = Path.home() / '.local' / 'envs' / 'oldman_isce2'
MIAPLPY = 'git+https://github.com/insarlab/MiaplPy.git@0d5f4b0cf0de46658c31b850a25c32a5258b773d'

if not YML.exists():
    raise FileNotFoundError(f'Put oldman_isce2_env.yml in this folder first: {HERE}')
MAMBA = shutil.which('mamba') or '/opt/conda/condabin/mamba'
print('mamba      :', MAMBA)
print('yml        :', YML)
print('environment:', PREFIX)

mamba      : /opt/conda/bin/mamba
yml        : /home/jovyan/Oldman Dam/oldman_isce2_env.yml
environment: /home/jovyan/.local/envs/oldman_isce2


## 1. Create the environment (20–40 min)
If the environment already exists, this cell skips it. To rebuild it from scratch, delete the folder `~/.local/envs/oldman_isce2` first.

In [2]:
if (PREFIX / 'bin' / 'python').exists():
    print('The environment already exists - skipped.')
else:
    !{MAMBA} env create -f "{YML}" --prefix "{PREFIX}" --yes

[+] 0.0s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                     ✔ Done (8.3 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (1.1 sec)

Resolving Environment                                                                    ✔ Done (10.0 sec)

Transaction

  Prefix: /home/jovyan/.local/envs/oldman_isce2

  Updating specs:

   - _openmp_mutex=4.5=20_gnu
   - _python_abi3_support=1.0=hd8ed1ab_3
   - affine=3.0.1=pyhecae5ae_0
   - aiohappyeyeballs=2.7.1=pyhd8ed1ab_0
   - aiohttp=3.1

## 2. Install MiaplPy into the environment (2–5 min)

In [3]:
# MiaplPy has two small C (Cython) parts. They are compiled with the compiler that is inside the
# environment, so nothing else needs to be installed on the server.
PY = PREFIX / 'bin' / 'python'
CC = PREFIX / 'bin' / 'x86_64-conda-linux-gnu-cc'
env = dict(os.environ)
env['PATH'] = str(PREFIX / 'bin') + os.pathsep + env['PATH']
env['CC'] = str(CC)
env['LDSHARED'] = f"{CC} -pthread -shared -B {PREFIX}/compiler_compat -Wl,-rpath,{PREFIX}/lib -L{PREFIX}/lib"
r = subprocess.run([str(PY), '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', MIAPLPY],
                   env=env, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
print('MiaplPy installed' if r.returncode == 0 else 'MiaplPy install FAILED - send me the text above')

  Cloning https://github.com/insarlab/MiaplPy.git (to revision 0d5f4b0cf0de46658c31b850a25c32a5258b773d) to /tmp/pip-req-build-ks78inky
  Resolved https://github.com/insarlab/MiaplPy.git to commit 0d5f4b0cf0de46658c31b850a25c32a5258b773d
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for miaplpy: filename=miaplpy-0.2.1-cp311-cp311-linux_x86_64.whl size=1187505 sha256=4a2b19aa3b3a36319c6414a44fa1503dc33fdb1151562f4524f6d49217c7bb08
  Stored in directory: /home/jovyan/.cache/pip/wheels/fd/7f/6d/c87bb8a03b7fc67c962480d3209cfc171084e44600b8dcb267
Successfully built miaplpy
   Running command git clone --filter=blob:none --quiet https://github.com/insarlab/MiaplPy.git /tmp/pip-req-build-ks78inky
  Running command git rev-parse -q --verify '0d5f4b0cf0de46658c31b850a25c32a5258b773d^{commit}'
  0d5f4b0cf0de46658c31b850a25c32a5258b773d

MiaplPy installed


## 3. Register the environment as a notebook kernel

In [4]:
!"{PY}" -m ipykernel install --user --name oldman_isce2 --display-name oldman_isce2

Installed kernelspec oldman_isce2 in /home/jovyan/.local/share/jupyter/kernels/oldman_isce2


## 4. Check that everything works

In [5]:
ENV = dict(os.environ)
stack = PREFIX / 'share' / 'isce2'
isce_home = next((PREFIX / 'lib').glob('python3*/site-packages/isce'))
ENV['PATH'] = os.pathsep.join([str(stack / 'topsStack'), str(isce_home / 'applications'), str(PREFIX / 'bin'), ENV['PATH']])
ENV['PYTHONPATH'] = str(stack)
check = ('import isce, mintpy, miaplpy, burst2safe, s1_orbits, sardem, asf_search; '
         'from miaplpy.lib import utils; '
         'print("isce", isce.__version__, "| mintpy", mintpy.__version__, "| asf_search", asf_search.__version__)')
r = subprocess.run([str(PY), '-c', check], env=ENV, capture_output=True, text=True)
print(r.stdout or r.stderr[-2000:])
for tool in ['stackSentinel.py', 'SentinelWrapper.py', 'prep_slc_isce.py', 'load_slc_geometry.py', 'burst2safe']:
    print(f'{tool:22s}', shutil.which(tool, path=ENV['PATH']) or 'NOT FOUND')

Using default ISCE Path: /home/jovyan/.local/envs/oldman_isce2/lib/python3.11/site-packages/isce
Using default MintPy Path: /home/jovyan/.local/envs/oldman_isce2/lib/python3.11/site-packages
2026-11-08 15:25:46,794 - h5py._conv - DEBUG - Creating converter from 7 to 5
2026-11-08 15:25:46,794 - h5py._conv - DEBUG - Creating converter from 5 to 7
2026-11-08 15:25:46,794 - h5py._conv - DEBUG - Creating converter from 7 to 5
2026-11-08 15:25:46,794 - h5py._conv - DEBUG - Creating converter from 5 to 7
isce 2.6.5 | mintpy 1.6.4 | asf_search 14.1.0

stackSentinel.py       /home/jovyan/.local/envs/oldman_isce2/share/isce2/topsStack/stackSentinel.py
SentinelWrapper.py     /home/jovyan/.local/envs/oldman_isce2/share/isce2/topsStack/SentinelWrapper.py
prep_slc_isce.py       /home/jovyan/.local/envs/oldman_isce2/bin/prep_slc_isce.py
load_slc_geometry.py   /home/jovyan/.local/envs/oldman_isce2/bin/load_slc_geometry.py
burst2safe             /home/jovyan/.local/envs/oldman_isce2/bin/burst2safe


**Done.** You should see an `isce 2.6.5 | mintpy 1.6.4 | ...` line and a path for every tool.

Next, open **`01_Oldman_ISCE2_coregistered_SLC_stack.ipynb`**. Choose the kernel **`oldman_isce2`** at the top right (or Kernel → Change Kernel).